# 🧬 FDA FAERS 2025 — AI Pharmacovigilance Pipeline

**Author:** Sami Bahig | Data Scientist | MILA · CRCHUM

**Certification:** UC San Diego — Big Data Specialization

End-to-end pipeline: 28M rows · Signal detection (PRR/ROR) · NLP · K-Means clustering · Plotly visualizations

In [ ]:
# -*- coding: utf-8 -*-
"""Big_data.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1HmPNfs2YRUMoXL74MXqjPmOUKNz4Fceq
"""

## ╗

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║        FAERS PIPELINE COMPLET - 4 TRIMESTRES - GOOGLE COLAB     ║
# ╚══════════════════════════════════════════════════════════════════╝

import pandas as pd
import os
import glob
import zipfile
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from google.colab import files

# ─────────────────────────────────────────────────────────────────────
# 1. UPLOAD DES 4 FICHIERS ZIP EN MÊME TEMPS
# ─────────────────────────────────────────────────────────────────────
print("📂 Sélectionnez vos 4 fichiers faers_ascii_*.zip en même temps")
print("   (maintenez Ctrl ou Cmd pour sélectionner plusieurs fichiers)\n")
uploaded = files.upload()
print(f"\n✅ {len(uploaded)} fichier(s) uploadé(s) :")
for f in uploaded.keys():
    print(f"   → {f}")

# ─────────────────────────────────────────────────────────────────────
# 2. EXTRACTION DE TOUS LES ZIPS → dossier ASCII unique
# ─────────────────────────────────────────────────────────────────────
ascii_dir = '/content/ASCII'
os.makedirs(ascii_dir, exist_ok=True)

print(f"\n📦 Extraction de tous les zips vers {ascii_dir}...")
for filename in uploaded.keys():
    zip_path = f"/content/{filename}"
    print(f"   Extraction : {filename}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        for member in z.namelist():
            if member.upper().endswith('.TXT'):
                member_filename = os.path.basename(member)
                target_path = os.path.join(ascii_dir, member_filename)
                with z.open(member) as src, open(target_path, 'wb') as dst:
                    dst.write(src.read())

txt_files_found = sorted(os.listdir(ascii_dir))
print(f"\n✅ Extraction terminée — {len(txt_files_found)} fichiers .txt disponibles :")
for f in txt_files_found:
    print(f"   → {f}")

# ─────────────────────────────────────────────────────────────────────
# 3. CHARGEMENT DES FICHIE

Faers plotly linkedin · PY
Copy

# ============================================================
#  🎨 VISUALISATIONS PLOTLY — LinkedIn Ready
#  Thème sombre professionnel, export PNG haute résolution
# ============================================================

# !pip install plotly kaleido -q  # décommenter si besoin

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from google.colab import files

# Palette de couleurs cohérente
COLORS = {
    'primary'   : '#00D4FF',   # cyan vif
    'secondary' : '#FF6B6B',   # rouge corail
    'accent'    : '#FFD93D',   # jaune doré
    'green'     : '#6BCB77',   # vert succès
    'purple'    : '#C77DFF',   # violet
    'bg'        : '#0D1117',   # fond très sombre
    'bg2'       : '#161B22',   # fond carte
    'text'      : '#E6EDF3',   # texte clair
    'grid'      : '#21262D',   # grille
}

TEMPLATE = dict(
    layout=go.Layout(
        paper_bgcolor=COLORS['bg'],
        plot_bgcolor=COLORS['bg2'],
        font=dict(family='Inter, Arial', color=COLORS['text'], size=13),
        title_font=dict(size=20, color=COLORS['primary']),
        legend=dict(bgcolor=COLORS['bg2'], bordercolor=COLORS['grid']),
        xaxis=dict(gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
        yaxis=dict(gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
    )
)

def save_and_download(fig, filename, width=1200, height=700):
    """Sauvegarde en PNG haute résolution et télécharge."""
    fig.write_image(filename, width=width, height=height, scale=2)
    files.download(filename)
    print(f'✅ {filename} téléchargé')


# ============================================================
# 📊 VIZ 1 — Treemap : Top 30 médicaments suspects
# ============================================================
print('Création Viz 1 — Treemap médicaments...')

df_drugs = pd.read_csv('viz_top_drugs.csv')

fig1 = px.treemap(
    df_drugs,
    path=['drugname'],
    values='count',
    color='count',
    color_continuous_scale=['#1a1a2e', '#16213e', '#0f3460', '#00D4FF', '#FFD93D'],
    title='🔬 Top 30 Médicaments Suspects — FAERS 2025 (1.6M rapports)',
)
fig1.update_traces(
    textinfo='label+value',
    textfont=dict(size=13, family='Inter'),
    hovertemplate='<b>%{label}</b><br>Rapports: %{value:,}<extra></extra>',
)
fig1.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    coloraxis_showscale=False,
    margin=dict(t=60, l=10, r=10, b=10),
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=0, y=-0.02, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E')
    )]
)
fig1.show()
save_and_download(fig1, 'linkedin_01_treemap_drugs.png', width=1200, height=750)


# ============================================================
# 📊 VIZ 2 — Bar chart horizontal : Top 20 réactions
# ============================================================
print('\nCréation Viz 2 — Réactions adverses...')

df_reac_viz = pd.read_csv('viz_top_reactions.csv').head(20).sort_values('count')

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=df_reac_viz['count'],
    y=df_reac_viz['reaction'],
    orientation='h',
    marker=dict(
        color=df_reac_viz['count'],
        colorscale=[[0, '#1a1a2e'], [0.5, '#00D4FF'], [1, '#FFD93D']],
        line=dict(color=COLORS['bg'], width=0.5),
    ),
    hovertemplate='<b>%{y}</b><br>Rapports: %{x:,}<extra></extra>',
))
fig2.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title='⚠️ Top 20 Réactions Adverses (MedDRA PT) — FAERS 2025',
    xaxis_title='Nombre de rapports',
    yaxis_title='',
    margin=dict(t=70, l=280, r=40, b=50),
    height=650,
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=1, y=-0.08, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='right'
    )]
)
fig2.show()
save_and_download(fig2, 'linkedin_02_top_reactions.png', width=1200, height=680)


# ============================================================
# 📊 VIZ 3 — Donut : Outcomes (gravité)
# ============================================================
print('\nCréation Viz 3 — Outcomes...')

df_outc_viz = pd.read_csv('viz_outcomes.csv')
outc_colors = [COLORS['secondary'], '#FF9F1C', COLORS['primary'],
               COLORS['purple'], COLORS['green'], '#FFD93D', '#8B949E']

fig3 = go.Figure(go.Pie(
    labels=df_outc_viz['outcome'],
    values=df_outc_viz['count'],
    hole=0.55,
    marker=dict(colors=outc_colors, line=dict(color=COLORS['bg'], width=3)),
    hovertemplate='<b>%{label}</b><br>%{value:,} rapports<br>%{percent}<extra></extra>',
    textinfo='label+percent',
    textfont=dict(size=12),
))
fig3.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title='🏥 Gravité des Effets Indésirables — FAERS 2025',
    annotations=[
        dict(text='1.2M<br>outcomes', x=0.5, y=0.5, font_size=18,
             showarrow=False, font=dict(color=COLORS['text'])),
        dict(text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
             x=0.5, y=-0.12, xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='center')
    ],
    height=600,
    showlegend=True,
    legend=dict(x=1.0, y=0.5),
)
fig3.show()
save_and_download(fig3, 'linkedin_03_outcomes.png', width=1100, height=650)


# ============================================================
# 📊 VIZ 4 — Carte choroplèthe : Pays reporters
# ============================================================
print('\nCréation Viz 4 — Carte mondiale...')

df_countries = pd.read_csv('viz_countries.csv')
# Correction EU → pas un code ISO valide, on le garde en annotation
df_countries_map = df_countries[df_countries['country_code'] != 'EU'].copy()

fig4 = px.choropleth(
    df_countries_map,
    locations='country_code',
    color='count',
    hover_name='country_name',
    hover_data={'count': ':,', 'country_code': False},
    color_continuous_scale=['#0D1117', '#0f3460', '#00D4FF', '#FFD93D'],
    title='🌍 Répartition Mondiale des Rapports FAERS 2025',
)
fig4.update_geos(
    showframe=False,
    showcoastlines=True,
    coastlinecolor='#21262D',
    showland=True, landcolor='#161B22',
    showocean=True, oceancolor='#0D1117',
    showlakes=False,
    projection_type='natural earth',
)
fig4.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    coloraxis_colorbar=dict(
        title='Rapports',
        tickfont=dict(color=COLORS['text']),
        titlefont=dict(color=COLORS['text']),
    ),
    margin=dict(t=60, l=0, r=0, b=30),
    height=550,
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=0.5, y=-0.05, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='center'
    )]
)
fig4.show()
save_and_download(fig4, 'linkedin_04_world_map.png', width=1300, height=600)


# ============================================================
# 📊 VIZ 5 — Scatter : Signaux VEDOLIZUMAB (PRR vs n)
# ============================================================
print('\nCréation Viz 5 — Signaux pharmacovigilance...')

df_sig = pd.read_csv('viz_signals_vedolizumab.csv')
df_sig_plot = df_sig[df_sig['n'] >= 5].copy()
df_sig_plot['size'] = np.clip(df_sig_plot['n'] / 50, 4, 30)
df_sig_plot['label_short'] = df_sig_plot['reaction'].str[:40]

confirmed = df_sig_plot[df_sig_plot['signal_confirmed'] == True]
not_conf  = df_sig_plot[df_sig_plot['signal_confirmed'] == False]

fig5 = go.Figure()

# Points non significatifs
fig5.add_trace(go.Scatter(
    x=not_conf['n'], y=not_conf['PRR'],
    mode='markers',
    name='Non significatif',
    marker=dict(color=COLORS['primary'], size=6, opacity=0.4,
                line=dict(color=COLORS['bg'], width=0.5)),
    hovertemplate='<b>%{customdata}</b><br>n=%{x}<br>PRR=%{y:.2f}<extra></extra>',
    customdata=not_conf['reaction'],
))

# Points signaux confirmés
fig5.add_trace(go.Scatter(
    x=confirmed['n'], y=confirmed['PRR'],
    mode='markers+text',
    name='🔴 Signal confirmé',
    marker=dict(
        color=COLORS['secondary'], size=confirmed['size']*1.5,
        opacity=0.85, line=dict(color=COLORS['accent'], width=1.5),
        symbol='circle',
    ),
    text=confirmed[confirmed['n'] > 200]['label_short'],
    textposition='top right',
    textfont=dict(size=9, color=COLORS['accent']),
    hovertemplate='<b>%{customdata}</b><br>n=%{x:,}<br>PRR=%{y:.2f}<br>ROR=%{marker.color:.2f}<extra></extra>',
    customdata=confirmed['reaction'],
))

# Lignes de seuil
fig5.add_hline(y=2, line=dict(color=COLORS['accent'], dash='dash', width=1.5),
               annotation_text='Seuil PRR=2', annotation_font_color=COLORS['accent'])
fig5.add_vline(x=3, line=dict(color='#8B949E', dash='dot', width=1),
               annotation_text='n min=3', annotation_font_color='#8B949E')

fig5.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title='🔍 Détection de Signaux Pharmacovigilance — VEDOLIZUMAB (FAERS 2025)<br>'
          '<sup>716 signaux confirmés sur 2524 réactions analysées (PRR≥2, Chi²≥4, ROR IC95inf>1)</sup>',
    xaxis=dict(title='Nombre de rapports (n)', type='log',
               gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
    yaxis=dict(title='PRR (Proportional Reporting Ratio)', type='log',
               gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
    legend=dict(x=0.7, y=0.95),
    height=680,
    margin=dict(t=90, l=80, r=40, b=60),
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=1, y=-0.1, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='right'
    )]
)
fig5.show()
save_and_download(fig5, 'linkedin_05_signals_vedolizumab.png', width=1300, height=720)


# ============================================================
# 📊 VIZ 6 — Dashboard résumé (4 panels en 1 image)
# ============================================================
print('\nCréation Viz 6 — Dashboard résumé...')

df_sex     = pd.read_csv('viz_sex.csv')
df_age     = pd.read_csv('viz_age_groups.csv').sort_values('count', ascending=True)
df_quarter = pd.read_csv('viz_quarters.csv')
df_top5    = pd.read_csv('viz_top_drugs.csv').head(10).sort_values('count', ascending=True)

fig6 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '👤 Répartition par Sexe',
        '📅 Rapports par Quarter',
        '🎂 Groupe d\'âge',
        '💊 Top 10 Médicaments Suspects',
    ),
    specs=[
        [{'type': 'pie'},    {'type': 'xy'}],
        [{'type': 'xy'},     {'type': 'xy'}],
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.10,
)

# Panel 1 : Sexe (donut)
sex_colors = [COLORS['secondary'], COLORS['primary'], '#8B949E']
fig6.add_trace(go.Pie(
    labels=df_sex['sex_label'], values=df_sex['count'],
    hole=0.5, marker_colors=sex_colors,
    textinfo='label+percent', textfont_size=11,
    showlegend=False,
    hovertemplate='<b>%{label}</b>: %{value:,}<extra></extra>',
), row=1, col=1)

# Panel 2 : Quarters (bar)
fig6.add_trace(go.Bar(
    x=df_quarter['quarter'], y=df_quarter['count'],
    marker_color=COLORS['primary'],
    hovertemplate='%{x}: %{y:,} rapports<extra></extra>',
    showlegend=False,
), row=1, col=2)

# Panel 3 : Âge (bar horizontal)
fig6.add_trace(go.Bar(
    x=df_age['count'], y=df_age['age_group'],
    orientation='h',
    marker_color=COLORS['purple'],
    hovertemplate='<b>%{y}</b>: %{x:,}<extra></extra>',
    showlegend=False,
), row=2, col=1)

# Panel 4 : Top 10 médicaments (bar horizontal)
fig6.add_trace(go.Bar(
    x=df_top5['count'], y=df_top5['drugname'],
    orientation='h',
    marker=dict(
        color=df_top5['count'],
        colorscale=[[0, '#0f3460'], [1, '#FFD93D']],
    ),
    hovertemplate='<b>%{y}</b>: %{x:,}<extra></extra>',
    showlegend=False,
), row=2, col=2)

fig6.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title=dict(
        text='📊 FAERS 2025 — Tableau de Bord Pharmacovigilance<br>'
             '<sup>1.6M rapports · Q1-Q4 2025 · FDA Adverse Event Reporting System</sup>',
        font=dict(size=18, color=COLORS['primary']),
    ),
    height=800,
    margin=dict(t=100, l=60, r=40, b=60),
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | Analyse : Python · Pandas · Plotly | @VotreNom',
        x=0.5, y=-0.06, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='center'
    )]
)

# Style axes
for row, col in [(1,2),(2,1),(2,2)]:
    fig6.update_xaxes(gridcolor=COLORS['grid'], row=row, col=col)
    fig6.update_yaxes(gridcolor=COLORS['grid'], row=row, col=col)

fig6.show()
save_and_download(fig6, 'linkedin_06_dashboard.png', width=1400, height=850)

print('\n🎉 Toutes les visualisations sont téléchargées !')
print('\n📋 Récapitulatif des fichiers LinkedIn :')
print('  linkedin_01_treemap_drugs.png      → Slide 1 : Médicaments suspects')
print('  linkedin_02_top_reactions.png      → Slide 2 : Réactions adverses')
print('  linkedin_03_outcomes.png           → Slide 3 : Gravité')
print('  linkedin_04_world_map.png          → Slide 4 : Carte mondiale')
print('  linkedin_05_signals_vedolizumab.png → Slide 5 : Signal detection')
print('  linkedin_06_dashboard.png          → Slide 6 : Dashboard résumé')

import plotly.express as px
import plotly.graph_objects as go

# Get top 10 countries from df_demo
df_countries_demo = df_demo['reporter_country'].value_counts().head(10).reset_index()
df_countries_demo.columns = ['country', 'count']

fig_country = px.bar(
    df_countries_demo,
    x='count',
    y='country',
    orientation='h',
    title='Top 10 Reporting Countries (from df_demo)',
    color='count',
    color_continuous_scale=px.colors.sequential.Teal,
)

# Apply a similar template/theme for consistency
COLORS = {
    'primary'   : '#00D4FF',
    'secondary' : '#FF6B6B',
    'accent'    : '#FFD93D',
    'green'     : '#6BCB77',
    'purple'    : '#C77DFF',
    'bg'        : '#0D1117',
    'bg2'       : '#161B22',
    'text'      : '#E6EDF3',
    'grid'      : '#21262D',
}

TEMPLATE_VIZ = dict(
    layout=go.Layout(
        paper_bgcolor=COLORS['bg'],
        plot_bgcolor=COLORS['bg2'],
        font=dict(family='Inter, Arial', color=COLORS['text'], size=13),
        title_font=dict(size=20, color=COLORS['primary']),
        legend=dict(bgcolor=COLORS['bg2'], bordercolor=COLORS['grid']),
        xaxis=dict(gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
        yaxis=dict(gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
    )
)

# Get the base layout dictionary from the template
base_layout = TEMPLATE_VIZ['layout'].to_plotly_json()

# Extract and update yaxis properties
updated_yaxis = base_layout.get('yaxis', {})
updated_yaxis.update({
    'title': 'Country',
    'categoryorder': 'total ascending'
})

# Remove the original yaxis from base_layout to avoid duplication
if 'yaxis' in base_layout:
    del base_layout['yaxis']

fig_country.update_layout(
    **base_layout,
    title_font_size=18,
    xaxis_title='Number of Reports',
    yaxis=updated_yaxis, # Pass the single, complete yaxis dictionary
    margin=dict(t=70, l=100, r=40, b=50),
)

fig_country.show()

display(df_demo.head())

import zipfile, os, glob

ascii_dir = '/content/ASCII'
os.makedirs(ascii_dir, exist_ok=True)

# Cherche TOUS les zips uploadés (peu importe le nom avec (2))
zip_files = glob.glob('/content/*.zip')
print(f"Zips trouvés : {zip_files}")

for zp in zip_files:
    with zipfile.ZipFile(zp, 'r') as z:
        for member in z.namelist():
            fname = os.path.basename(member)
            if fname:  # ignore les dossiers
                z.extract(member, '/content/ASCII_tmp')

# Aplatir tous les .txt dans /content/ASCII/
import shutil
for root, dirs, files in os.walk('/content/ASCII_tmp'):
    for f in files:
        shutil.copy2(os.path.join(root, f), os.path.join(ascii_dir, f))

print("Fichiers disponibles :", os.listdir(ascii_dir))

# ============================================================
#  🚨 CORRECTIF RAM — À coller dans une nouvelle cellule
#  Interrompez d'abord la cellule en cours (bouton ⏹)
#  puis exécutez CE code uniquement
# ============================================================

import gc
import psutil
import pandas as pd
import numpy as np

# ── Étape A : Libérer tout ce qui traîne en mémoire ─────────────
for var in ['data', 'df_demo_dedup', 'X', 'sample']:
    if var in dir():
        del var
gc.collect()

ram = psutil.virtual_memory()
print(f"💾 RAM disponible après nettoyage : {ram.available/1e9:.1f} GB")

# ── Étape B : Réduire agressivement les colonnes ────────────────
print("\nRéduction des colonnes DEMO...")
demo_keep = ['primaryid', 'caseversion', 'age', 'age_grp', 'sex',
             'reporter_country', 'fda_dt', 'rept_cod', 'occp_cod']
demo_keep = [c for c in demo_keep if c in df_demo.columns]
df_demo_slim = (
    df_demo[demo_keep]
    .sort_values('caseversion', ascending=False)
    .drop_duplicates(subset='primaryid', keep='first')
    .copy()
)
del df_demo
gc.collect()
print(f"  DEMO slim : {df_demo_slim.shape}  "
      f"({df_demo_slim.memory_usage(deep=True).sum()/1e6:.0f} MB)")

print("Réduction des colonnes DRUG...")
drug_keep = ['primaryid', 'drug_seq', 'role_cod', 'drugname', 'route', 'dechal']
drug_keep = [c for c in drug_keep if c in df_drug.columns]
df_drug_slim = df_drug[drug_keep].copy()
del df_drug
gc.collect()
print(f"  DRUG slim : {df_drug_slim.shape}  "
      f"({df_drug_slim.memory_usage(deep=True).sum()/1e6:.0f} MB)")

print("Réduction des colonnes REAC...")
reac_keep = ['primaryid', 'pt']
reac_keep = [c for c in reac_keep if c in df_reac.columns]
df_reac_slim = df_reac[reac_keep].copy()
del df_reac
gc.collect()
print(f"  REAC slim : {df_reac_slim.shape}  "
      f"({df_reac_slim.memory_usage(deep=True).sum()/1e6:.0f} MB)")

ram = psutil.virtual_memory()
print(f"\n💾 RAM disponible avant merge : {ram.available/1e9:.1f} GB")

# ── Étape C : Optimiser les types avant le merge ────────────────
print("\nOptimisation des types (int64 → int32 / category)...")

for df, name in [(df_demo_slim,'DEMO'), (df_drug_slim,'DRUG'), (df_reac_slim,'REAC')]:
    for col in df.select_dtypes(include='int64').columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include='object').columns:
        if df[col].nunique() < 500:
            df[col] = df[col].astype('category')
    print(f"  {name} : {df.memory_usage(deep=True).sum()/1e6:.0f} MB")

gc.collect()
ram = psutil.virtual_memory()
print(f"\n💾 RAM disponible après optimisation types : {ram.available/1e9:.1f} GB")

# ── Étape D : Merge en deux passes avec surveillance RAM ─────────
print("\nMerge DEMO + DRUG...")
data = pd.merge(df_demo_slim, df_drug_slim, on='primaryid', how='inner')
del df_demo_slim, df_drug_slim
gc.collect()
ram = psutil.virtual_memory()
print(f"  Shape DEMO+DRUG : {data.shape}  "
      f"({data.memory_usage(deep=True).sum()/1e9:.2f} GB)  "
      f"| RAM restante : {ram.available/1e9:.1f} GB")

print("Merge + REAC...")
data = pd.merge(data, df_reac_slim, on='primaryid', how='inner')
del df_reac_slim
gc.collect()
ram = psutil.virtual_memory()
print(f"  Shape finale : {data.shape}  "
      f"({data.memory_usage(deep=True).sum()/1e9:.2f} GB)  "
      f"| RAM restante : {ram.available/1e9:.1f} GB")

# ── Étape E : Renommage ──────────────────────────────────────────
rename_map = {
    'age' : 'patientage',
    'sex' : 'patientsex',
    'pt'  : 'reactionmeddrapt',
}
data = data.rename(columns={k: v for k, v in rename_map.items() if k in data.columns})

print(f"\n✅ Merge terminé !")
print(f"   Colonnes : {data.columns.tolist()}")
print(data.head(3))

# ============================================================
#  FAERS 2025 — Analyse Complète optimisée pour Google Colab Pro
#  RAM cible : ~12-15 GB peak (confortable sur les ~25 GB dispo)
#  Objectifs : Statistiques · NLP · Signal Detection (PRR/ROR)
# ============================================================

# ============================================================
# ÉTAPE 0 — Vérification RAM & installations
# ============================================================
import psutil
ram = psutil.virtual_memory()
print(f"✅ RAM totale    : {ram.total/1e9:.1f} GB")
print(f"✅ RAM disponible: {ram.available/1e9:.1f} GB")

# Installer wordcloud si absent
try:
    from wordcloud import WordCloud
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'wordcloud', '-q'])

import zipfile, os, glob, shutil, gc, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
from collections import Counter

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)


# ============================================================
# ÉTAPE 1 — Upload & Extraction des ZIP FAERS
# ============================================================
from google.colab import files

ASCII_DIR = '/content/ASCII'
os.makedirs(ASCII_DIR, exist_ok=True)

print('\n📂 Sélectionnez vos 4 fichiers faers_ascii_*.zip')
print('   (maintenez Ctrl ou Cmd pour sélectionner plusieurs fichiers)\n')
uploaded = files.upload()

zip_paths = []
for fname in uploaded.keys():
    dst = f'/content/{fname}'
    if os.path.exists(fname) and fname != dst:
        shutil.move(fname, dst)
    zip_paths.append(dst)

print(f'\n✅ {len(zip_paths)} zip(s) détecté(s)')

for zp in zip_paths:
    print(f'   📦 Extraction : {os.path.basename(zp)}')
    try:
        with zipfile.ZipFile(zp, 'r') as z:
            for member in z.namelist():
                fname_only = os.path.basename(member)
                if fname_only == '':
                    continue
                target = os.path.join(ASCII_DIR, fname_only)
                if not os.path.exists(target):  # évite les doublons (2), (3)...
                    with z.open(member) as src_f, open(target, 'wb') as dst_f:
                        shutil.copyfileobj(src_f, dst_f)
    except zipfile.BadZipFile:
        print(f'   ❌ Fichier invalide : {zp}')

txt_files = sorted(glob.glob(os.path.join(ASCII_DIR, '*.txt')))
print(f'\n✅ {len(txt_files)} fichiers .txt disponibles dans {ASCII_DIR}')
for f in txt_files:
    print(f'   → {os.path.basename(f)}')


# ============================================================
# ÉTAPE 2 — Chargement & Concaténation
# ============================================================

def load_files(pattern, ascii_dir=ASCII_DIR, usecols=None):
    """Charge tous les .txt correspondant au pattern."""
    paths = sorted(glob.glob(os.path.join(ascii_dir, pattern)))
    print(f'  {pattern}: {[os.path.basename(p) for p in paths]}')
    dfs = []
    for p in paths:
        try:
            df = pd.read_csv(p, sep='$', encoding='utf-8',
                             low_memory=False, usecols=usecols)
        except UnicodeDecodeError:
            df = pd.read_csv(p, sep='$', encoding='latin1',
                             low_memory=False, usecols=usecols)
        except ValueError:
            # usecols non disponibles dans ce fichier, charger tout
            try:
                df = pd.read_csv(p, sep='$', encoding='utf-8', low_memory=False)
            except UnicodeDecodeError:
                df = pd.read_csv(p, sep='$', encoding='latin1', low_memory=False)
        df.columns = df.columns.str.lower().str.strip()
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print('\nChargement des fichiers (colonnes slim pour économiser la RAM)...')

# Charger uniquement les colonnes nécessaires
df_demo = load_files('DEMO*.txt', usecols=[
    'PRIMARYID','CASEID','CASEVERSION','AGE','AGE_COD','AGE_GRP',
    'SEX','WT','WT_COD','REPORTER_COUNTRY','OCCR_COUNTRY',
    'OCCP_COD','FDA_DT','REPT_COD','I_F_COD','REPT_DT'
])
df_drug = load_files('DRUG*.txt', usecols=[
    'PRIMARYID','CASEID','DRUG_SEQ','ROLE_COD','DRUGNAME',
    'PROD_AI','ROUTE','DECHAL','RECHAL','DOSE_AMT','DOSE_UNIT','DOSE_FORM'
])
df_reac = load_files('REAC*.txt', usecols=['PRIMARYID','CASEID','PT'])
df_outc = load_files('OUTC*.txt', usecols=['PRIMARYID','CASEID','OUTC_COD'])
df_rpsr = load_files('RPSR*.txt', usecols=['PRIMARYID','CASEID','RPSR_COD'])
df_indi = load_files('INDI*.txt', usecols=['PRIMARYID','CASEID','INDI_DRUG_SEQ','INDI_PT'])
df_ther = load_files('THER*.txt', usecols=['PRIMARYID','CASEID','DSG_DRUG_SEQ','START_DT','END_DT','DUR','DUR_COD'])

print(f'\n✅ Dimensions chargées :')
for name, df in [('DEMO',df_demo),('DRUG',df_drug),('REAC',df_reac),
                 ('OUTC',df_outc),('RPSR',df_rpsr),('INDI',df_indi),('THER',df_ther)]:
    mem = df.memory_usage(deep=True).sum()/1e6
    print(f'   {name} : {df.shape}  ({mem:.0f} MB)')

ram = psutil.virtual_memory()
print(f'\n💾 RAM disponible après chargement : {ram.available/1e9:.1f} GB')


# ============================================================
# ÉTAPE 3 — Merge optimisé (anti-crash RAM)
# ============================================================

# 3.1 Dédupliquer DEMO : garder la version la plus récente par primaryid
print('\nDéduplication DEMO (garde caseversion max par primaryid)...')
df_demo_dedup = (
    df_demo
    .sort_values('caseversion', ascending=False)
    .drop_duplicates(subset='primaryid', keep='first')
    .copy()
)
print(f'  DEMO : {df_demo.shape[0]:,} → {df_demo_dedup.shape[0]:,} lignes uniques')
del df_demo
gc.collect()

# 3.2 Merge DEMO + DRUG
print('\nMerge DEMO + DRUG...')
data = pd.merge(df_demo_dedup, df_drug,
                on='primaryid', how='inner',
                suffixes=('_demo','_drug'))
print(f'  Shape après DEMO+DRUG : {data.shape}')
del df_demo_dedup, df_drug
gc.collect()

# 3.3 Merge + REAC
print('Merge + REAC...')
data = pd.merge(data, df_reac, on='primaryid', how='inner')
print(f'  Shape finale : {data.shape}')
del df_reac
gc.collect()

# 3.4 Renommage
rename_map = {
    'age'     : 'patientage',
    'sex'     : 'patientsex',
    'pt'      : 'reactionmeddrapt',
    'i_f_cod' : 'i_f_code',
}
data = data.rename(columns={k: v for k, v in rename_map.items() if k in data.columns})

ram = psutil.virtual_memory()
print(f'\n✅ Merge terminé')
print(f'   Shape finale       : {data.shape}')
print(f'   RAM du DataFrame   : {data.memory_usage(deep=True).sum()/1e9:.2f} GB')
print(f'   RAM Colab restante : {ram.available/1e9:.1f} GB')
print(f'\nColonnes : {data.columns.tolist()}')
print(data.head(3))


# ============================================================
# ÉTAPE 4 — Statistiques Descriptives
# ============================================================
print('\n' + '='*60)
print('ÉTAPE 4 — STATISTIQUES DESCRIPTIVES')
print('='*60)

# 4.1 Distribution par sexe
print('\n--- 4.1 Distribution par sexe ---')
sex_counts = data['patientsex'].value_counts(dropna=False)
print(sex_counts)
sex_counts.plot(kind='bar', color=['steelblue','salmon','grey'],
                title='Rapports par sexe', rot=0)
plt.ylabel('Nombre de rapports')
plt.tight_layout()
plt.show()

# 4.2 Groupe d'âge
print('\n--- 4.2 Distribution par groupe d\'âge ---')
age_grp_map = {
    'N':'Neonate','I':'Infant','C':'Child',
    'T':'Adolescent','A':'Adult','E':'Elderly'
}
if 'age_grp' in data.columns:
    ag = data['age_grp'].map(age_grp_map).value_counts(dropna=False)
    print(ag)
    ag.dropna().plot(kind='bar', color='teal',
                     title="Rapports par groupe d'âge", rot=30)
    plt.ylabel('Nombre de rapports')
    plt.tight_layout()
    plt.show()

# 4.3 Top 20 médicaments suspects
print('\n--- 4.3 Top 20 médicaments suspects (rôle PS) ---')
top_drugs = (
    data[data['role_cod'] == 'PS']['drugname']
    .str.upper().str.strip()
    .value_counts().head(20)
)
print(top_drugs)
top_drugs.plot(kind='barh', title='Top 20 médicaments suspects', color='steelblue')
plt.xlabel('Nombre de rapports')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 4.4 Top 20 réactions
print('\n--- 4.4 Top 20 réactions adverses ---')
top_reac = data['reactionmeddrapt'].value_counts().head(20)
print(top_reac)
top_reac.plot(kind='barh', color='coral',
              title='Top 20 réactions adverses (MedDRA PT)')
plt.xlabel('Nombre de rapports')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 4.5 Outcomes
print('\n--- 4.5 Distribution des outcomes ---')
outc_map = {
    'DE':'Décès','LT':'Pronostic vital','HO':'Hospitalisation',
    'DS':'Handicap','CA':'Anomalie congénitale',
    'RI':'Intervention requise','OT':'Autre grave'
}
outc_counts = df_outc['outc_cod'].map(outc_map).value_counts()
print(outc_counts)
outc_counts.plot(kind='pie', autopct='%1.1f%%',
                 title='Outcomes 2025 — tous quarters', startangle=90)
plt.ylabel('')
plt.tight_layout()
plt.show()

# 4.6 Évolution par quarter
print('\n--- 4.6 Rapports par quarter ---')
data['quarter'] = pd.to_datetime(
    data['fda_dt'].astype(str), format='%Y%m%d', errors='coerce'
).dt.to_period('Q')
q_counts = data['quarter'].value_counts().sort_index()
print(q_counts)
q_counts.plot(kind='bar', color='mediumseagreen',
              title='Rapports par quarter', rot=0)
plt.ylabel('Nombre de rapports')
plt.tight_layout()
plt.show()

# 4.7 Pays reporters (Top 15)
print('\n--- 4.7 Top 15 pays reporters ---')
if 'reporter_country' in data.columns:
    top_countries = data['reporter_country'].value_counts().head(15)
    print(top_countries)
    top_countries.plot(kind='barh', color='mediumpurple',
                       title='Top 15 pays reporters')
    plt.xlabel('Nombre de rapports')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


# ============================================================
# ÉTAPE 5 — NLP sur les réactions MedDRA
# ============================================================
print('\n' + '='*60)
print('ÉTAPE 5 — NLP SUR LES RÉACTIONS MEDDRA')
print('='*60)

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def clean_tokens(text):
    tokens = word_tokenize(str(text).lower())
    return [w for w in tokens if w.isalpha() and w not in stop_words and len(w) > 2]

# Sélection & nettoyage
nlp_cols = ['drugname','reactionmeddrapt','patientsex','patientage']
df_nlp = data[nlp_cols].dropna().copy()
df_nlp['tokens'] = df_nlp['reactionmeddrapt'].apply(clean_tokens)

print(f'Lignes pour NLP : {len(df_nlp):,}')
print('\nExemple :')
print(df_nlp[['reactionmeddrapt','tokens']].head(5))

# Fréquence des termes
all_tokens = [t for tokens in df_nlp['tokens'] for t in tokens]
top_tok = pd.Series(Counter(all_tokens)).sort_values(ascending=False).head(20)
print('\nTop 20 termes MedDRA :')
print(top_tok)

# Nuage de mots
try:
    from wordcloud import WordCloud
    wc = WordCloud(width=900, height=400, background_color='white',
                   colormap='RdYlBu', max_words=150)
    wc.generate(' '.join(all_tokens))
    plt.figure(figsize=(14, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Nuage de mots — Termes MedDRA les plus fréquents (2025)')
    plt.tight_layout()
    plt.show()
except ImportError:
    top_tok.plot(kind='barh', title='Top 20 termes MedDRA')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


# ============================================================
# ÉTAPE 6 — TF-IDF + K-Means Clustering
# ============================================================
print('\n' + '='*60)
print('ÉTAPE 6 — TF-IDF + CLUSTERING DES RÉACTIONS')
print('='*60)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Échantillon de 50k pour la performance
sample = df_nlp.sample(min(50000, len(df_nlp)), random_state=42).copy()
corpus = sample['reactionmeddrapt'].str.lower()

tfidf = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2))
X = tfidf.fit_transform(corpus)
print(f'Matrice TF-IDF : {X.shape}')

N_CLUSTERS = 5
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, max_iter=300)
sample['cluster'] = km.fit_predict(X)

print('\nDistribution par cluster :')
print(sample['cluster'].value_counts().sort_index())

feature_names = tfidf.get_feature_names_out()
print('\n=== Termes caractéristiques par cluster ===')
for i in range(N_CLUSTERS):
    center = km.cluster_centers_[i]
    top_terms = [feature_names[j] for j in center.argsort()[-8:][::-1]]
    print(f'  Cluster {i}: {", ".join(top_terms)}')

sample['cluster'].value_counts().sort_index().plot(
    kind='bar', color='mediumpurple',
    title='Distribution des clusters de réactions', rot=0)
plt.xlabel('Cluster')
plt.ylabel('Nombre de rapports')
plt.tight_layout()
plt.show()

# Réactions dominantes par cluster
print('\n--- Réaction la plus fréquente par cluster ---')
for i in range(N_CLUSTERS):
    top_r = sample[sample['cluster'] == i]['reactionmeddrapt'].value_counts().head(3)
    print(f'  Cluster {i}: {top_r.index.tolist()}')


# ============================================================
# ÉTAPE 7 — Détection de Signaux : PRR & ROR
# ============================================================
print('\n' + '='*60)
print('ÉTAPE 7 — DÉTECTION DE SIGNAUX (PRR & ROR)')
print('='*60)

signal_data = data[data['role_cod'] == 'PS'][['drugname','reactionmeddrapt']].dropna().copy()
signal_data['drugname']          = signal_data['drugname'].str.upper().str.strip()
signal_data['reactionmeddrapt']  = signal_data['reactionmeddrapt'].str.strip()

N_total = len(signal_data)
print(f'Paires drogue-réaction (suspects PS) : {N_total:,}')

top10 = signal_data['drugname'].value_counts().head(10)
print(f'\nTop 10 médicaments suspects :')
print(top10)

# ──────────────────────────────────────────────────────────────────
# ➡️  MODIFIEZ ICI LE MÉDICAMENT À ANALYSER
DRUG_OF_INTEREST = top10.index[0]
# Exemple : DRUG_OF_INTEREST = 'OZEMPIC'
# ──────────────────────────────────────────────────────────────────
print(f'\n➡️  Analyse de signaux pour : {DRUG_OF_INTEREST}')


def compute_prr_ror(drug, df, min_n=3):
    """Calcule PRR et ROR (avec IC95%) pour toutes les réactions d'un médicament."""
    results = []
    drug_mask = df['drugname'] == drug

    for reaction in df.loc[drug_mask, 'reactionmeddrapt'].unique():
        reac_mask = df['reactionmeddrapt'] == reaction
        a = int((drug_mask  &  reac_mask).sum())
        b = int((drug_mask  & ~reac_mask).sum())
        c = int((~drug_mask &  reac_mask).sum())
        d = int((~drug_mask & ~reac_mask).sum())

        if a < min_n or b == 0 or c == 0:
            continue

        prr    = (a / (a + b)) / (c / (c + d))
        ror    = (a * d) / (b * c)
        se     = np.sqrt(1/a + 1/b + 1/c + 1/d)
        ror_lo = np.exp(np.log(ror) - 1.96 * se)
        ror_hi = np.exp(np.log(ror) + 1.96 * se)

        # Chi-carré simple
        n = a + b + c + d
        e_a = ((a + b) * (a + c)) / n
        chi2 = ((a - e_a) ** 2) / e_a if e_a > 0 else 0

        results.append({
            'reaction'      : reaction,
            'n'             : a,
            'PRR'           : round(prr, 3),
            'ROR'           : round(ror, 3),
            'ROR_IC95_low'  : round(ror_lo, 3),
            'ROR_IC95_high' : round(ror_hi, 3),
            'Chi2'          : round(chi2, 2),
            'signal_PRR'    : (prr >= 2) and (chi2 >= 4),
            'signal_ROR'    : ror_lo > 1,
            'signal_confirmed': (prr >= 2) and (chi2 >= 4) and (ror_lo > 1),
        })

    return pd.DataFrame(results).sort_values('PRR', ascending=False).reset_index(drop=True)


df_signals = compute_prr_ror(DRUG_OF_INTEREST, signal_data)

confirmed = df_signals[df_signals['signal_confirmed']]
print(f'\n=== Signaux confirmés (PRR≥2 + Chi²≥4 + ROR IC95inf>1) ===')
print(f'    Médicament : {DRUG_OF_INTEREST}')
print(f'    Total réactions analysées : {len(df_signals)}')
print(f'    Signaux confirmés         : {len(confirmed)}')
print()
print(confirmed[['reaction','n','PRR','ROR','ROR_IC95_low','ROR_IC95_high','Chi2']].head(20).to_string())


# ============================================================
# ÉTAPE 8 — Visualisation des Signaux
# ============================================================
print('\n' + '='*60)
print('ÉTAPE 8 — VISUALISATION DES SIGNAUX')
print('='*60)

# Forest plot — Top 20 par PRR
top20 = df_signals.head(20).copy()
colors = ['red' if r else 'steelblue' for r in top20['signal_confirmed']]

fig, ax = plt.subplots(figsize=(11, 8))
for i, (_, row) in enumerate(top20.iterrows()):
    ax.plot([row['ROR_IC95_low'], row['ROR_IC95_high']], [i, i],
            color=colors[i], lw=2, alpha=0.8)
    ax.plot(row['ROR'], i, 'o', color=colors[i], ms=7)
ax.axvline(x=1, color='black', linestyle='--', lw=1, label='ROR=1')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['reaction'], fontsize=8)
ax.set_xlabel('ROR (IC 95%)')
ax.set_title(f'Forest Plot — {DRUG_OF_INTEREST}\n'
             f'(rouge = signal confirmé : PRR≥2, Chi²≥4, ROR IC95inf>1)')
red_patch  = mpatches.Patch(color='red',       label='Signal confirmé')
blue_patch = mpatches.Patch(color='steelblue', label='Non significatif')
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.show()

# Bubble chart
if not confirmed.empty:
    plot_df = confirmed.head(30)
    fig, ax = plt.subplots(figsize=(13, 6))
    sc = ax.scatter(
        plot_df['n'], plot_df['PRR'],
        s=(plot_df['n'].clip(upper=300) * 2),
        c=plot_df['ROR'], cmap='YlOrRd',
        alpha=0.75, edgecolors='black', linewidths=0.5
    )
    for _, row in plot_df.iterrows():
        ax.annotate(row['reaction'][:35], (row['n'], row['PRR']),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(3, 3), textcoords='offset points')
    plt.colorbar(sc, label='ROR')
    ax.axhline(y=2, color='red', linestyle='--', lw=1.5, label='Seuil PRR=2')
    ax.set_xlabel('Nombre de rapports (n)')
    ax.set_ylabel('PRR')
    ax.set_title(f'Bubble Chart — Signaux confirmés : {DRUG_OF_INTEREST}')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Tableau récap des top signaux
    print('\n📋 Tableau récapitulatif des signaux confirmés :')
    print(confirmed[['reaction','n','PRR','ROR','ROR_IC95_low','ROR_IC95_high','Chi2']]
          .head(20)
          .to_string(index=False))
else:
    print(f'ℹ️  Aucun signal confirmé pour {DRUG_OF_INTEREST}.')
    print('   Essayez un autre médicament en modifiant DRUG_OF_INTEREST ci-dessus.')


# ============================================================
# ÉTAPE 9 — Export des résultats
# ============================================================
print('\n' + '='*60)
print('ÉTAPE 9 — EXPORT')
print('='*60)

from google.colab import files

# Signaux CSV
out_signals = f'signals_{DRUG_OF_INTEREST.replace(" ","_")}_2025.csv'
df_signals.to_csv(out_signals, index=False)
print(f'✅ Signaux exportés : {out_signals}')
files.download(out_signals)

# Données NLP CSV
df_nlp[['drugname','reactionmeddrapt','patientsex','patientage']].to_csv(
    'faers_2025_nlp.csv', index=False)
print('✅ Données NLP exportées : faers_2025_nlp.csv')
files.download('faers_2025_nlp.csv')

# Statistiques descriptives CSV
stats_summary = pd.DataFrame({
    'total_reports'     : [data['primaryid'].nunique()],
    'total_drug_reaction_pairs' : [len(data)],
    'top_drug'          : [top_drugs.index[0]],
    'top_reaction'      : [top_reac.index[0]],
    'quarters_covered'  : ['2025Q1-Q4'],
})
stats_summary.to_csv('faers_2025_stats_summary.csv', index=False)
print('✅ Résumé statistiques exporté : faers_2025_stats_summary.csv')
files.download('faers_2025_stats_summary.csv')

ram = psutil.virtual_memory()
print(f'\n💾 RAM finale disponible : {ram.available/1e9:.1f} GB')
print('\n🎉 Analyse FAERS 2025 complète terminée !')

# ============================================================
#  🚨 CORRECTIF RAM — À coller dans une nouvelle cellule
#  Interrompez d'abord la cellule en cours (bouton ⏹)
#  puis exécutez CE code uniquement
# ============================================================

import gc
import psutil
import pandas as pd
import numpy as np

# ── Étape A : Libérer tout ce qui traîne en mémoire ─────────────
for var in ['data', 'df_demo_dedup', 'X', 'sample']:
    if var in dir():
        del var
gc.collect()

ram = psutil.virtual_memory()
print(f"💾 RAM disponible après nettoyage : {ram.available/1e9:.1f} GB")

# ── Étape B : Réduire agressivement les colonnes ────────────────
print("\nRéduction des colonnes DEMO...")
demo_keep = ['primaryid', 'caseversion', 'age', 'age_grp', 'sex',
             'reporter_country', 'fda_dt', 'rept_cod', 'occp_cod']
demo_keep = [c for c in demo_keep if c in df_demo.columns]
df_demo_slim = (
    df_demo[demo_keep]
    .sort_values('caseversion', ascending=False)
    .drop_duplicates(subset='primaryid', keep='first')
    .copy()
)
del df_demo
gc.collect()
print(f"  DEMO slim : {df_demo_slim.shape}  "
      f"({df_demo_slim.memory_usage(deep=True).sum()/1e6:.0f} MB)")

print("Réduction des colonnes DRUG...")
drug_keep = ['primaryid', 'drug_seq', 'role_cod', 'drugname', 'route', 'dechal']
drug_keep = [c for c in drug_keep if c in df_drug.columns]
df_drug_slim = df_drug[drug_keep].copy()
del df_drug
gc.collect()
print(f"  DRUG slim : {df_drug_slim.shape}  "
      f"({df_drug_slim.memory_usage(deep=True).sum()/1e6:.0f} MB)")

print("Réduction des colonnes REAC...")
reac_keep = ['primaryid', 'pt']
reac_keep = [c for c in reac_keep if c in df_reac.columns]
df_reac_slim = df_reac[reac_keep].copy()
del df_reac
gc.collect()
print(f"  REAC slim : {df_reac_slim.shape}  "
      f"({df_reac_slim.memory_usage(deep=True).sum()/1e6:.0f} MB)")

ram = psutil.virtual_memory()
print(f"\n💾 RAM disponible avant merge : {ram.available/1e9:.1f} GB")

# ── Étape C : Optimiser les types avant le merge ────────────────
print("\nOptimisation des types (int64 → int32 / category)...")

for df, name in [(df_demo_slim,'DEMO'), (df_drug_slim,'DRUG'), (df_reac_slim,'REAC')]:
    for col in df.select_dtypes(include='int64').columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include='object').columns:
        if df[col].nunique() < 500:
            df[col] = df[col].astype('category')
    print(f"  {name} : {df.memory_usage(deep=True).sum()/1e6:.0f} MB")

gc.collect()
ram = psutil.virtual_memory()
print(f"\n💾 RAM disponible après optimisation types : {ram.available/1e9:.1f} GB")

# ── Étape D : Merge en deux passes avec surveillance RAM ─────────
print("\nMerge DEMO + DRUG...")
data = pd.merge(df_demo_slim, df_drug_slim, on='primaryid', how='inner')
del df_demo_slim, df_drug_slim
gc.collect()
ram = psutil.virtual_memory()
print(f"  Shape DEMO+DRUG : {data.shape}  "
      f"({data.memory_usage(deep=True).sum()/1e9:.2f} GB)  "
      f"| RAM restante : {ram.available/1e9:.1f} GB")

print("Merge + REAC...")
data = pd.merge(data, df_reac_slim, on='primaryid', how='inner')
del df_reac_slim
gc.collect()
ram = psutil.virtual_memory()
print(f"  Shape finale : {data.shape}  "
      f"({data.memory_usage(deep=True).sum()/1e9:.2f} GB)  "
      f"| RAM restante : {ram.available/1e9:.1f} GB")

# ── Étape E : Renommage ──────────────────────────────────────────
rename_map = {
    'age' : 'patientage',
    'sex' : 'patientsex',
    'pt'  : 'reactionmeddrapt',
}
data = data.rename(columns={k: v for k, v in rename_map.items() if k in data.columns})

print(f"\n✅ Merge terminé !")
print(f"   Colonnes : {data.columns.tolist()}")
print(data.head(3))

#============================================================
#  🔄 RECHARGEMENT COMPLET — colonnes slim dès la lecture
#  Coller dans une nouvelle cellule et exécuter
# ============================================================

import pandas as pd
import numpy as np
import gc
import psutil
import glob
import os

ASCII_DIR = '/content/ASCII'

def load_slim(pattern, usecols_lower):
    """Charge les fichiers en ne gardant QUE les colonnes demandées (en minuscules)."""
    paths = sorted(glob.glob(os.path.join(ASCII_DIR, pattern)))
    print(f'  {pattern} → {[os.path.basename(p) for p in paths]}')
    dfs = []
    for p in paths:
        # Lire d'abord juste le header pour trouver les noms exacts
        try:
            header = pd.read_csv(p, sep='$', encoding='utf-8', nrows=0)
        except UnicodeDecodeError:
            header = pd.read_csv(p, sep='$', encoding='latin1', nrows=0)
        header.columns = header.columns.str.lower().str.strip()

        # Filtrer les colonnes disponibles
        cols_ok = [c for c in usecols_lower if c in header.columns]

        try:
            df = pd.read_csv(p, sep='$', encoding='utf-8',
                             usecols=cols_ok, low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(p, sep='$', encoding='latin1',
                             usecols=cols_ok, low_memory=False)

        df.columns = df.columns.str.lower().str.strip()

        # Optimiser les types pour économiser la RAM
        for col in df.select_dtypes(include='int64').columns:
            df[col] = df[col].astype('int32')
        for col in df.select_dtypes(include='object').columns:
            if df[col].nunique() < 1000:
                df[col] = df[col].astype('category')

        dfs.append(df)

    result = pd.concat(dfs, ignore_index=True)
    mem = result.memory_usage(deep=True).sum() / 1e6
    print(f'    → shape: {result.shape}  ({mem:.0f} MB)')
    return result

ram = psutil.virtual_memory()
print(f'💾 RAM disponible au départ : {ram.available/1e9:.1f} GB\n')

print('Chargement slim...')

df_demo = load_slim('DEMO*.txt', [
    'primaryid', 'caseversion', 'age', 'age_grp', 'sex',
    'reporter_country', 'fda_dt', 'rept_cod', 'occp_cod'
])

df_drug = load_slim('DRUG*.txt', [
    'primaryid', 'drug_seq', 'role_cod', 'drugname', 'route', 'dechal'
])

df_reac = load_slim('REAC*.txt', [
    'primaryid', 'pt'
])

df_outc = load_slim('OUTC*.txt', [
    'primaryid', 'outc_cod'
])

ram = psutil.virtual_memory()
print(f'\n💾 RAM disponible après chargement : {ram.available/1e9:.1f} GB')

# ── Déduplication DEMO ───────────────────────────────────────────
print('\nDéduplication DEMO...')
df_demo = (
    df_demo
    .sort_values('caseversion', ascending=False)
    .drop_duplicates(subset='primaryid', keep='first')
    .reset_index(drop=True)
)
gc.collect()
print(f'  DEMO dédupliqué : {df_demo.shape}')

# ── Merge DEMO + DRUG ────────────────────────────────────────────
print('\nMerge DEMO + DRUG...')
data = pd.merge(df_demo, df_drug, on='primaryid', how='inner')
del df_demo, df_drug
gc.collect()
ram = psutil.virtual_memory()
print(f'  Shape : {data.shape}  '
      f'({data.memory_usage(deep=True).sum()/1e9:.2f} GB)  '
      f'| RAM restante : {ram.available/1e9:.1f} GB')

# ── Merge + REAC ─────────────────────────────────────────────────
print('\nMerge + REAC...')
data = pd.merge(data, df_reac, on='primaryid', how='inner')
del df_reac
gc.collect()
ram = psutil.virtual_memory()
print(f'  Shape finale : {data.shape}  '
      f'({data.memory_usage(deep=True).sum()/1e9:.2f} GB)  '
      f'| RAM restante : {ram.available/1e9:.1f} GB')

# ── Renommage ─────────────────────────────────────────────────────
data = data.rename(columns={
    'age' : 'patientage',
    'sex' : 'patientsex',
    'pt'  : 'reactionmeddrapt',
})

print(f'\n✅ Données prêtes !')
print(f'   Colonnes : {data.columns.tolist()}')
print(data.head(3))

# ============================================================
#  🔧 CORRECTIF STRATÉGIE — REAC séparé (évite l'explosion)
#  Coller dans une nouvelle cellule APRÈS le rechargement slim
#  (df_demo, df_drug, df_reac, df_outc sont déjà en mémoire)
# ============================================================

import pandas as pd
import numpy as np
import gc
import psutil

# ── Libérer le data explosé ──────────────────────────────────────
print("Nettoyage du data explosé...")
try:
    del data
    gc.collect()
except:
    pass

ram = psutil.virtual_memory()
print(f"💾 RAM disponible : {ram.available/1e9:.1f} GB\n")

# ── Recharger slim si nécessaire ─────────────────────────────────
import glob, os
ASCII_DIR = '/content/ASCII'

def load_slim(pattern, usecols_lower):
    paths = sorted(glob.glob(os.path.join(ASCII_DIR, pattern)))
    dfs = []
    for p in paths:
        try:
            header = pd.read_csv(p, sep='$', encoding='utf-8', nrows=0)
        except UnicodeDecodeError:
            header = pd.read_csv(p, sep='$', encoding='latin1', nrows=0)
        header.columns = header.columns.str.lower().str.strip()
        cols_ok = [c for c in usecols_lower if c in header.columns]
        try:
            df = pd.read_csv(p, sep='$', encoding='utf-8', usecols=cols_ok, low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(p, sep='$', encoding='latin1', usecols=cols_ok, low_memory=False)
        df.columns = df.columns.str.lower().str.strip()
        for col in df.select_dtypes(include='int64').columns:
            df[col] = df[col].astype('int32')
        for col in df.select_dtypes(include='object').columns:
            if df[col].nunique() < 1000:
                df[col] = df[col].astype('category')
        dfs.append(df)
    result = pd.concat(dfs, ignore_index=True)
    print(f'  {pattern}: {result.shape}  ({result.memory_usage(deep=True).sum()/1e6:.0f} MB)')
    return result

# Recharger si variables perdues
if 'df_demo' not in dir():
    print("Rechargement df_demo...")
    df_demo = load_slim('DEMO*.txt', ['primaryid','caseversion','age','age_grp','sex',
                                       'reporter_country','fda_dt','rept_cod','occp_cod'])
if 'df_drug' not in dir():
    print("Rechargement df_drug...")
    df_drug = load_slim('DRUG*.txt', ['primaryid','drug_seq','role_cod','drugname','route','dechal'])
if 'df_reac' not in dir():
    print("Rechargement df_reac...")
    df_reac = load_slim('REAC*.txt', ['primaryid','pt'])
if 'df_outc' not in dir():
    print("Rechargement df_outc...")
    df_outc = load_slim('OUTC*.txt', ['primaryid','outc_cod'])
if 'df_indi' not in dir(): # Add df_indi to the reload logic
    print("Rechargement df_indi...")
    df_indi = load_slim('INDI*.txt', ['primaryid','indi_pt'])

# ── STRATÉGIE CORRECTE ───────────────────────────────────────────
# data_demo_drug  : 1 ligne par (rapport × drogue)   → stats médicaments
# df_reac         : 1 ligne par (rapport × réaction)  → stats réactions
# On NE merge PAS les deux ensemble globalement

print("\n" + "="*60)
print("CONSTRUCTION DES TABLES DE TRAVAIL")
print("="*60)

# Table 1 : DEMO + DRUG (1 ligne par rapport×drogue)
print("\nConstruction data_demo_drug (DEMO + DRUG)...")
df_demo_dedup = (
    df_demo
    .sort_values('caseversion', ascending=False)
    .drop_duplicates(subset='primaryid', keep='first')
    .reset_index(drop=True)
)
data_demo_drug = pd.merge(df_demo_dedup, df_drug, on='primaryid', how='inner')
data_demo_drug = data_demo_drug.rename(columns={
    'age': 'patientage', 'sex': 'patientsex'
})
del df_demo_dedup
gc.collect()
ram = psutil.virtual_memory()
print(f"  Shape : {data_demo_drug.shape}  "
      f"({data_demo_drug.memory_usage(deep=True).sum()/1e9:.2f} GB)  "
      f"| RAM restante : {ram.available/1e9:.1f} GB")

# Table 2 : REAC renommé (gardé séparé)
df_reac = df_reac.rename(columns={'pt': 'reactionmeddrapt'})
print(f"\ndf_reac (séparé) : {df_reac.shape}  "
      f"({df_reac.memory_usage(deep=True).sum()/1e6:.0f} MB)")

# Table 3 : Pour signal detection uniquement → merge sur les suspects PS
print("\nConstruction signal_data (PS uniquement, DRUG + REAC)...")
ps_ids = data_demo_drug[data_demo_drug['role_cod'] == 'PS'][['primaryid','drugname']].copy()
ps_ids['drugname'] = ps_ids['drugname'].astype(str).str.upper().str.strip()

signal_data = pd.merge(ps_ids, df_reac, on='primaryid', how='inner')
signal_data['reactionmeddrapt'] = signal_data['reactionmeddrapt'].astype(str).str.strip()
del ps_ids
gc.collect()
ram = psutil.virtual_memory()
print(f"  Shape signal_data : {signal_data.shape}  "
      f"({signal_data.memory_usage(deep=True).sum()/1e9:.2f} GB)  "
      f"| RAM restante : {ram.available/1e9:.1f} GB")

print("\n✅ Tables de travail prêtes :")
print(f"   data_demo_drug : {data_demo_drug.shape}  (stats médicaments, démographie)")
print(f"   df_reac        : {df_reac.shape}  (stats réactions)")
print(f"   df_outc        : {df_outc.shape}  (outcomes/gravité)")
print(f"   signal_data    : {signal_data.shape}  (PRR/ROR signal detection)")
print(f"\nColonnes data_demo_drug : {data_demo_drug.columns.tolist()}")
print(data_demo_drug.head(3))


# ============================================================
# ÉTAPE 4 — Statistiques Descriptives
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

print('\n' + '='*60)
print('ÉTAPE 4 — STATISTIQUES DESCRIPTIVES')
print('='*60)

# 4.1 Distribution par sexe
print('\n--- 4.1 Distribution par sexe ---')
sex_counts = data_demo_drug['patientsex'].value_counts(dropna=False)
print(sex_counts)
sex_counts.plot(kind='bar', color=['steelblue','salmon','grey'],
                title='Rapports par sexe', rot=0)
plt.ylabel('Nombre de rapports')
plt.tight_layout(); plt.show()

# 4.2 Groupe d'âge
print('\n--- 4.2 Groupe d\'âge ---')
age_grp_map = {'N':'Neonate','I':'Infant','C':'Child',
               'T':'Adolescent','A':'Adult','E':'Elderly'}
if 'age_grp' in data_demo_drug.columns:
    ag = data_demo_drug['age_grp'].astype(str).map(age_grp_map).value_counts(dropna=True)
    print(ag)
    ag.plot(kind='bar', color='teal', title="Groupe d'âge", rot=30)
    plt.ylabel('Nombre de rapports')
plt.tight_layout(); plt.show()

# 4.3 Top 20 médicaments suspects
print('\n--- 4.3 Top 20 médicaments suspects (PS) ---')
top_drugs = (
    data_demo_drug[data_demo_drug['role_cod'] == 'PS']['drugname']
    .astype(str).str.upper().str.strip()
    .value_counts().head(20)
)
print(top_drugs)
top_drugs.plot(kind='barh', title='Top 20 médicaments suspects', color='steelblue')
plt.xlabel('Nombre de rapports')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

# 4.4 Top 20 réactions (depuis df_reac séparé)
print('\n--- 4.4 Top 20 réactions adverses ---')
top_reac = df_reac['reactionmeddrapt'].value_counts().head(20)
print(top_reac)
top_reac.plot(kind='barh', color='coral', title='Top 20 réactions adverses (MedDRA PT)')
plt.xlabel('Nombre de rapports')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

# 4.5 Outcomes
print('\n--- 4.5 Outcomes (gravité) ---')
outc_map = {'DE':'Décès','LT':'Pronostic vital','HO':'Hospitalisation',
            'DS':'Handicap','CA':'Anomalie congénitale',
            'RI':'Intervention requise','OT':'Autre grave'}
outc_counts = df_outc['outc_cod'].astype(str).map(outc_map).value_counts()
print(outc_counts)
outc_counts.plot(kind='pie', autopct='%1.1f%%',
                 title='Outcomes 2025 — tous quarters', startangle=90)
plt.ylabel(''); plt.tight_layout(); plt.show()

# 4.6 Rapports par quarter
print('\n--- 4.6 Rapports par quarter ---')
data_demo_drug['quarter'] = pd.to_datetime(
    data_demo_drug['fda_dt'].astype(str), format='%Y%m%d', errors='coerce'
).dt.to_period('Q')
q_counts = data_demo_drug['quarter'].value_counts().sort_index()
print(q_counts)
q_counts.plot(kind='bar', color='mediumseagreen',
              title='Rapports par quarter', rot=0)
plt.ylabel('Nombre de rapports')
plt.tight_layout(); plt.show()

# 4.7 Top 15 pays reporters
print('\n--- 4.7 Top 15 pays reporters ---')
if 'reporter_country' in data_demo_drug.columns:
    top_countries = data_demo_drug['reporter_country'].astype(str).value_counts().head(15)
    print(top_countries)
    top_countries.plot(kind='barh', color='mediumpurple',
                       title='Top 15 pays reporters')
    plt.xlabel('Nombre de rapports')
    plt.gca().invert_yaxis()
    plt.tight_layout(); plt.show()


# ============================================================
# ÉTAPE 5 — NLP sur les réactions MedDRA
# ============================================================
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('\n' + '='*60)
print('ÉTAPE 5 — NLP SUR LES RÉACTIONS MEDDRA')
print('='*60)

stop_words = set(stopwords.words('english'))

def clean_tokens(text):
    tokens = word_tokenize(str(text).lower())
    return [w for w in tokens if w.isalpha() and w not in stop_words and len(w) > 2]

# Travailler sur un échantillon de df_reac pour le NLP
nlp_sample = df_reac['reactionmeddrapt'].dropna().sample(
    min(200000, len(df_reac)), random_state=42
)
print(f'Échantillon NLP : {len(nlp_sample):,} réactions')

all_tokens = []
for text in nlp_sample:
    all_tokens.extend(clean_tokens(text))

top_tok = pd.Series(Counter(all_tokens)).sort_values(ascending=False).head(20)
print('\nTop 20 termes MedDRA :')
print(top_tok)

top_tok.plot(kind='barh', title='Top 20 termes MedDRA (tokens)', color='steelblue')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

# Nuage de mots
try:
    from wordcloud import WordCloud
    wc = WordCloud(width=900, height=400, background_color='white',
                   colormap='RdYlBu', max_words=150)
    wc.generate(' '.join(all_tokens))
    plt.figure(figsize=(14, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Nuage de mots — Termes MedDRA (2025)')
    plt.tight_layout(); plt.show()
except ImportError:
    print("Installez wordcloud : !pip install wordcloud")


# ============================================================
# ÉTAPE 6 — TF-IDF + K-Means Clustering
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

print('\n' + '='*60)
print('ÉTAPE 6 — TF-IDF + CLUSTERING')
print('='*60)

corpus = df_reac['reactionmeddrapt'].dropna().sample(
    min(50000, len(df_reac)), random_state=42
).str.lower().reset_index(drop=True)

tfidf = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2))
X = tfidf.fit_transform(corpus)
print(f'Matrice TF-IDF : {X.shape}')

N_CLUSTERS = 5
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, max_iter=300)
labels = km.fit_predict(X)

feature_names = tfidf.get_feature_names_out()
print('\nTermes caractéristiques par cluster :')
for i in range(N_CLUSTERS):
    center = km.cluster_centers_[i]
    top_terms = [feature_names[j] for j in center.argsort()[-8:][::-1]]
    n_in_cluster = (labels == i).sum()
    print(f'  Cluster {i} ({n_in_cluster:,} items): {", ".join(top_terms)}')

pd.Series(labels).value_counts().sort_index().plot(
    kind='bar', color='mediumpurple',
    title='Distribution des clusters de réactions', rot=0)
plt.xlabel('Cluster'); plt.ylabel('Nombre')
plt.tight_layout(); plt.show()

del X
gc.collect()


# ============================================================
# ÉTAPE 7 — Détection de Signaux : PRR & ROR
# ============================================================
print('\n' + '='*60)
print('ÉTAPE 7 — DÉTECTION DE SIGNAUX (PRR & ROR)')
print('='*60)

N_total = len(signal_data)
print(f'Paires drogue-réaction (suspects PS) : {N_total:,}')

top10 = signal_data['drugname'].value_counts().head(10)
print(f'\nTop 10 médicaments suspects :')
print(top10)

# ── MODIFIEZ ICI LE MÉDICAMENT ──────────────────────────────────
DRUG_OF_INTEREST = top10.index[0]
# DRUG_OF_INTEREST = 'OZEMPIC'
# ────────────────────────────────────────────────────────────────
print(f'\n➡️  Analyse : {DRUG_OF_INTEREST}')

def compute_prr_ror(drug, df, min_n=3):
    results = []
    drug_mask = df['drugname'] == drug
    for reaction in df.loc[drug_mask, 'reactionmeddrapt'].unique():
        reac_mask = df['reactionmeddrapt'] == reaction
        a = int(( drug_mask &  reac_mask).sum())
        b = int(( drug_mask & ~reac_mask).sum())
        c = int((~drug_mask &  reac_mask).sum())
        d = int((~drug_mask & ~reac_mask).sum())
        if a < min_n or b == 0 or c == 0:
            continue
        prr    = (a/(a+b)) / (c/(c+d))
        ror    = (a*d) / (b*c)
        se     = np.sqrt(1/a + 1/b + 1/c + 1/d)
        ror_lo = np.exp(np.log(ror) - 1.96*se)
        ror_hi = np.exp(np.log(ror) + 1.96*se)
        n      = a+b+c+d
        e_a    = ((a+b)*(a+c))/n
        chi2   = ((a-e_a)**2)/e_a if e_a > 0 else 0
        results.append({
            'reaction': reaction, 'n': a,
            'PRR': round(prr,3), 'ROR': round(ror,3),
            'ROR_IC95_low': round(ror_lo,3), 'ROR_IC95_high': round(ror_hi,3),
            'Chi2': round(chi2,2),
            'signal_confirmed': (prr>=2) and (chi2>=4) and (ror_lo>1),
        })
    return pd.DataFrame(results).sort_values('PRR', ascending=False).reset_index(drop=True)

df_signals = compute_prr_ror(DRUG_OF_INTEREST, signal_data)
confirmed  = df_signals[df_signals['signal_confirmed']]

print(f'\nTotal réactions analysées : {len(df_signals)}')
print(f'Signaux confirmés         : {len(confirmed)}')
print(confirmed[['reaction','n','PRR','ROR','ROR_IC95_low','ROR_IC95_high','Chi2']].head(20).to_string())


# ============================================================
# ÉTAPE 8 — Visualisation des Signaux
# ============================================================
import matplotlib.patches as mpatches

print('\n' + '='*60)
print('ÉTAPE 8 — VISUALISATION DES SIGNAUX')
print('='*60)

top20 = df_signals.head(20).copy()
colors = ['red' if r else 'steelblue' for r in top20['signal_confirmed']]

fig, ax = plt.subplots(figsize=(11, 8))
for i, (_, row) in enumerate(top20.iterrows()):
    ax.plot([row['ROR_IC95_low'], row['ROR_IC95_high']], [i, i],
            color=colors[i], lw=2, alpha=0.8)
    ax.plot(row['ROR'], i, 'o', color=colors[i], ms=7)
ax.axvline(x=1, color='black', linestyle='--', lw=1)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['reaction'], fontsize=8)
ax.set_xlabel('ROR (IC 95%)')
ax.set_title(f'Forest Plot — {DRUG_OF_INTEREST}\n(rouge = PRR≥2, Chi²≥4, ROR IC95inf>1)')
ax.legend(handles=[
    mpatches.Patch(color='red',       label='Signal confirmé'),
    mpatches.Patch(color='steelblue', label='Non significatif'),
])
plt.tight_layout(); plt.show()

if not confirmed.empty:
    plot_df = confirmed.head(30)
    fig, ax = plt.subplots(figsize=(13, 6))
    sc = ax.scatter(plot_df['n'], plot_df['PRR'],
                    s=(plot_df['n'].clip(upper=300)*2),
                    c=plot_df['ROR'], cmap='YlOrRd',
                    alpha=0.75, edgecolors='black', linewidths=0.5)
    for _, row in plot_df.iterrows():
        ax.annotate(row['reaction'][:35], (row['n'], row['PRR']),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(3,3), textcoords='offset points')
    plt.colorbar(sc, label='ROR')
    ax.axhline(y=2, color='red', linestyle='--', lw=1.5, label='Seuil PRR=2')
    ax.set_xlabel('Nombre de rapports (n)')
    ax.set_ylabel('PRR')
    ax.set_title(f'Bubble Chart — Signaux confirmés : {DRUG_OF_INTEREST}')
    ax.legend(); plt.tight_layout(); plt.show()


# ============================================================
# ÉTAPE 9 — Export
# ============================================================
from google.colab import files

print('\n' + '='*60)
print('ÉTAPE 9 — EXPORT')
print('='*60)

out = f'signals_{DRUG_OF_INTEREST.replace(" ","_")}_2025.csv'
df_signals.to_csv(out, index=False)
print(f'✅ {out}')
files.download(out)

top_reac.to_csv('faers_2025_top_reactions.csv', header=True)
print('✅ faers_2025_top_reactions.csv')
files.download('faers_2025_top_reactions.csv')

top_drugs.to_csv('faers_2025_top_drugs.csv', header=True)
print('✅ faers_2025_top_drugs.csv')
files.download('faers_2025_top_drugs.csv')

ram = psutil.virtual_memory()
print(f'\n💾 RAM finale : {ram.available/1e9:.1f} GB disponible')
print('\n🎉 Analyse FAERS 2025 complète !')

#============================================================
#  📊 EXPORT CSV pour Plotly Studio
#  Coller dans une nouvelle cellule Colab
# ============================================================

from google.colab import files
import pandas as pd
import numpy as np

# ── 1. CSV : Médicaments suspects (Top 30) ───────────────────────
top_drugs_csv = (
    data_demo_drug[data_demo_drug['role_cod'] == 'PS']['drugname']
    .astype(str).str.upper().str.strip()
    .value_counts().head(30)
    .reset_index()
)
top_drugs_csv.columns = ['drugname', 'count']
top_drugs_csv.to_csv('viz_top_drugs.csv', index=False)
print(f'✅ viz_top_drugs.csv — {len(top_drugs_csv)} lignes')

# ── 2. CSV : Réactions adverses (Top 30) ────────────────────────
top_reac_csv = (
    df_reac['reactionmeddrapt']
    .value_counts().head(30)
    .reset_index()
)
top_reac_csv.columns = ['reaction', 'count']
top_reac_csv.to_csv('viz_top_reactions.csv', index=False)
print(f'✅ viz_top_reactions.csv — {len(top_reac_csv)} lignes')

# ── 3. CSV : Outcomes (gravité) ──────────────────────────────────
outc_map = {
    'DE':'Décès', 'LT':'Pronostic vital', 'HO':'Hospitalisation',
    'DS':'Handicap', 'CA':'Anomalie congénitale',
    'RI':'Intervention requise', 'OT':'Autre grave'
}
outc_csv = (
    df_outc['outc_cod'].astype(str).map(outc_map)
    .value_counts()
    .reset_index()
)
outc_csv.columns = ['outcome', 'count']
outc_csv.to_csv('viz_outcomes.csv', index=False)
print(f'✅ viz_outcomes.csv — {len(outc_csv)} lignes')

# ── 4. CSV : Pays reporters (Top 20) ────────────────────────────
country_csv = (
    data_demo_drug['reporter_country']
    .astype(str).value_counts().head(20)
    .reset_index()
)
country_csv.columns = ['country_code', 'count']
# Ajouter noms complets pour la lisibilité
country_names = {
    'US':'United States', 'CA':'Canada', 'EU':'European Union',
    'GB':'United Kingdom', 'JP':'Japan', 'CN':'China',
    'FR':'France', 'DE':'Germany', 'AU':'Australia', 'IT':'Italy',
    'ES':'Spain', 'BR':'Brazil', 'IN':'India', 'KR':'South Korea',
    'PL':'Poland', 'NL':'Netherlands', 'SE':'Sweden', 'CH':'Switzerland',
    'BE':'Belgium', 'MX':'Mexico'
}
country_csv['country_name'] = country_csv['country_code'].map(country_names).fillna(country_csv['country_code'])
country_csv.to_csv('viz_countries.csv', index=False)
print(f'✅ viz_countries.csv — {len(country_csv)} lignes')

# ── 5. CSV : Rapports par quarter ────────────────────────────────
quarter_csv = (
    data_demo_drug['quarter']
    .value_counts().sort_index()
    .reset_index()
)
quarter_csv.columns = ['quarter', 'count']
quarter_csv['quarter'] = quarter_csv['quarter'].astype(str)
quarter_csv = quarter_csv[quarter_csv['count'] > 100]  # exclure les outliers
quarter_csv.to_csv('viz_quarters.csv', index=False)
print(f'✅ viz_quarters.csv — {len(quarter_csv)} lignes')

# ── 6. CSV : Groupe d'âge ────────────────────────────────────────
age_grp_map = {
    'N':'Neonate', 'I':'Infant', 'C':'Child',
    'T':'Adolescent', 'A':'Adult', 'E':'Elderly'
}
age_csv = (
    data_demo_drug['age_grp'].astype(str).map(age_grp_map)
    .value_counts(dropna=True)
    .reset_index()
)
age_csv.columns = ['age_group', 'count']
age_csv = age_csv[age_csv['age_group'] != 'nan']
age_csv.to_csv('viz_age_groups.csv', index=False)
print(f'✅ viz_age_groups.csv — {len(age_csv)} lignes')

# ── 7. CSV : Signaux VEDOLIZUMAB (déjà exporté, reformaté) ──────
# Utiliser df_signals déjà calculé
signals_viz = df_signals[df_signals['n'] >= 5].copy()
signals_viz['signal_label'] = signals_viz['signal_confirmed'].map(
    {True: '🔴 Signal confirmé', False: '🔵 Non significatif'}
)
signals_viz.to_csv('viz_signals_vedolizumab.csv', index=False)
print(f'✅ viz_signals_vedolizumab.csv — {len(signals_viz)} lignes')

# ── 8. CSV : Sexe ────────────────────────────────────────────────
sex_csv = (
    data_demo_drug['patientsex']
    .value_counts(dropna=True)
    .reset_index()
)
sex_csv.columns = ['sex', 'count']
sex_map = {'F': 'Femme', 'M': 'Homme', 'UNK': 'Inconnu'}
sex_csv['sex_label'] = sex_csv['sex'].map(sex_map).fillna(sex_csv['sex'])
sex_csv.to_csv('viz_sex.csv', index=False)
print(f'✅ viz_sex.csv — {len(sex_csv)} lignes')

# ── Téléchargement de tous les CSV ──────────────────────────────
print('\n📥 Téléchargement...')
for fname in [
    'viz_top_drugs.csv',
    'viz_top_reactions.csv',
    'viz_outcomes.csv',
    'viz_countries.csv',
    'viz_quarters.csv',
    'viz_age_groups.csv',
    'viz_signals_vedolizumab.csv',
    'viz_sex.csv',
]:
    files.download(fname)
    print(f'   → {fname}')

print('\n🎉 Tous les CSV sont prêts pour Plotly Studio !')
print('\nVoici les visualisations recommandées par fichier :')
print('  viz_top_drugs.csv         → Bar chart horizontal (Treemap aussi)')
print('  viz_top_reactions.csv     → Bar chart horizontal')
print('  viz_outcomes.csv          → Pie chart / Donut')
print('  viz_countries.csv         → Choropleth map (carte mondiale)')
print('  viz_quarters.csv          → Line chart / Bar chart')
print('  viz_age_groups.csv        → Bar chart / Funnel')
print('  viz_signals_vedolizumab.csv → Scatter plot (PRR vs n, couleur = signal)')
print('  viz_sex.csv               → Pie chart / Donut')

!pip install kaleido -q

# ============================================================
#  🎨 VISUALISATIONS PLOTLY — LinkedIn Ready
#  Thème sombre professionnel, export PNG haute résolution
# ============================================================

# Installer kaleido pour l'exportation d'images si ce n'est pas déjà fait
!pip install kaleido -q

# Force reload plotly.io to ensure it detects newly installed kaleido
import importlib
import plotly.io as pio
importlib.reload(pio)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from google.colab import files

# Palette de couleurs cohérente
COLORS = {
    'primary'   : '#00D4FF',   # cyan vif
    'secondary' : '#FF6B6B',   # rouge corail
    'accent'    : '#FFD93D',   # jaune doré
    'green'     : '#6BCB77',   # vert succès
    'purple'    : '#C77DFF',   # violet
    'bg'        : '#0D1117',   # fond très sombre
    'bg2'       : '#161B22',   # fond carte
    'text'      : '#E6EDF3',   # texte clair
    'grid'      : '#21262D',   # grille
}

TEMPLATE = dict(
    layout=go.Layout(
        paper_bgcolor=COLORS['bg'],
        plot_bgcolor=COLORS['bg2'],
        font=dict(family='Inter, Arial', color=COLORS['text'], size=13),
        title_font=dict(size=20, color=COLORS['primary']),
        legend=dict(bgcolor=COLORS['bg2'], bordercolor=COLORS['grid']),
        xaxis=dict(gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
        yaxis=dict(gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
    )
)

def save_and_download(fig, filename, width=1200, height=700):
    """Sauvegarde en PNG haute résolution et télécharge."""
    fig.write_image(filename, width=width, height=height, scale=2)
    files.download(filename)
    print(f'✅ {filename} téléchargé')


# ============================================================
# 📊 VIZ 1 — Treemap : Top 30 médicaments suspects
# ============================================================
print('Création Viz 1 — Treemap médicaments...')

df_drugs = pd.read_csv('viz_top_drugs.csv')

fig1 = px.treemap(
    df_drugs,
    path=['drugname'],
    values='count',
    color='count',
    color_continuous_scale=['#1a1a2e', '#16213e', '#0f3460', '#00D4FF', '#FFD93D'],
    title='🔬 Top 30 Médicaments Suspects — FAERS 2025 (1.6M rapports)',
)
fig1.update_traces(
    textinfo='label+value',
    textfont=dict(size=13, family='Inter'),
    hovertemplate='<b>%{label}</b><br>Rapports: %{value:,}<extra></extra>',
)
fig1.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    coloraxis_showscale=False,
    margin=dict(t=60, l=10, r=10, b=10),
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=0, y=-0.02, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E')
    )]
)
fig1.show()
save_and_download(fig1, 'linkedin_01_treemap_drugs.png', width=1200, height=750)


# ============================================================
# 📊 VIZ 2 — Bar chart horizontal : Top 20 réactions
# ============================================================
print('\nCréation Viz 2 — Réactions adverses...')

df_reac_viz = pd.read_csv('viz_top_reactions.csv').head(20).sort_values('count')

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=df_reac_viz['count'],
    y=df_reac_viz['reaction'],
    orientation='h',
    marker=dict(
        color=df_reac_viz['count'],
        colorscale=[[0, '#1a1a2e'], [0.5, '#00D4FF'], [1, '#FFD93D']],
        line=dict(color=COLORS['bg'], width=0.5),
    ),
    hovertemplate='<b>%{y}</b><br>Rapports: %{x:,}<extra></extra>',
))
fig2.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title='⚠️ Top 20 Réactions Adverses (MedDRA PT) — FAERS 2025',
    xaxis_title='Nombre de rapports',
    yaxis_title='',
    margin=dict(t=70, l=280, r=40, b=50),
    height=650,
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=1, y=-0.08, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='right'
    )]
)
fig2.show()
save_and_download(fig2, 'linkedin_02_top_reactions.png', width=1200, height=680)


# ============================================================
# 📊 VIZ 3 — Donut : Outcomes (gravité)
# ============================================================
print('\nCréation Viz 3 — Outcomes...')

df_outc_viz = pd.read_csv('viz_outcomes.csv')
outc_colors = [COLORS['secondary'], '#FF9F1C', COLORS['primary'],
               COLORS['purple'], COLORS['green'], '#FFD93D', '#8B949E']

fig3 = go.Figure(go.Pie(
    labels=df_outc_viz['outcome'],
    values=df_outc_viz['count'],
    hole=0.55,
    marker=dict(colors=outc_colors, line=dict(color=COLORS['bg'], width=3)),
    hovertemplate='<b>%{label}</b><br>%{value:,} rapports<br>%{percent}<extra></extra>',
    textinfo='label+percent',
    textfont=dict(size=12),
))
fig3.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title='🏥 Gravité des Effets Indésirables — FAERS 2025',
    annotations=[
        dict(text='1.2M<br>outcomes', x=0.5, y=0.5, font_size=18,
             showarrow=False, font=dict(color=COLORS['text'])),
        dict(text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
             x=0.5, y=-0.12, xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='center')
    ],
    height=600,
    showlegend=True,
    legend=dict(x=1.0, y=0.5),
)
fig3.show()
save_and_download(fig3, 'linkedin_03_outcomes.png', width=1100, height=650)


# ============================================================
# 📊 VIZ 4 — Carte choroplèthe : Pays reporters
# ============================================================
print('\nCréation Viz 4 — Carte mondiale...')

df_countries = pd.read_csv('viz_countries.csv')
# Correction EU → pas un code ISO valide, on le garde en annotation
df_countries_map = df_countries[df_countries['country_code'] != 'EU'].copy()

fig4 = px.choropleth(
    df_countries_map,
    locations='country_code',
    color='count',
    hover_name='country_name',
    hover_data={'count': ':,', 'country_code': False},
    color_continuous_scale=['#0D1117', '#0f3460', '#00D4FF', '#FFD93D'],
    title='🌍 Répartition Mondiale des Rapports FAERS 2025',
)
fig4.update_geos(
    showframe=False,
    showcoastlines=True,
    coastlinecolor='#21262D',
    showland=True, landcolor='#161B22',
    showocean=True, oceancolor='#0D1117',
    showlakes=False,
    projection_type='natural earth',
)
fig4.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    coloraxis_colorbar=dict(
        title='Rapports',
        tickfont=dict(color=COLORS['text']),
        titlefont=dict(color=COLORS['text']),
    ),
    margin=dict(t=60, l=0, r=0, b=30),
    height=550,
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=0.5, y=-0.05, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='center'
    )]
)
fig4.show()
save_and_download(fig4, 'linkedin_04_world_map.png', width=1300, height=600)


# ============================================================
# 📊 VIZ 5 — Scatter : Signaux VEDOLIZUMAB (PRR vs n)
# ============================================================
print('\nCréation Viz 5 — Signaux pharmacovigilance...')

df_sig = pd.read_csv('viz_signals_vedolizumab.csv')
df_sig_plot = df_sig[df_sig['n'] >= 5].copy()
df_sig_plot['size'] = np.clip(df_sig_plot['n'] / 50, 4, 30)
df_sig_plot['label_short'] = df_sig_plot['reaction'].str[:40]

confirmed = df_sig_plot[df_sig_plot['signal_confirmed'] == True]
not_conf  = df_sig_plot[df_sig_plot['signal_confirmed'] == False]

fig5 = go.Figure()

# Points non significatifs
fig5.add_trace(go.Scatter(
    x=not_conf['n'], y=not_conf['PRR'],
    mode='markers',
    name='Non significatif',
    marker=dict(color=COLORS['primary'], size=6, opacity=0.4,
                line=dict(color=COLORS['bg'], width=0.5)),
    hovertemplate='<b>%{customdata}</b><br>n=%{x}<br>PRR=%{y:.2f}<extra></extra>',
    customdata=not_conf['reaction'],
))

# Points signaux confirmés
fig5.add_trace(go.Scatter(
    x=confirmed['n'], y=confirmed['PRR'],
    mode='markers+text',
    name='🔴 Signal confirmé',
    marker=dict(
        color=COLORS['secondary'], size=confirmed['size']*1.5,
        opacity=0.85, line=dict(color=COLORS['accent'], width=1.5),
        symbol='circle',
    ),
    text=confirmed[confirmed['n'] > 200]['label_short'],
    textposition='top right',
    textfont=dict(size=9, color=COLORS['accent']),
    hovertemplate='<b>%{customdata}</b><br>n=%{x:,}<br>PRR=%{y:.2f}<br>ROR=%{marker.color:.2f}<extra></extra>',
    customdata=confirmed['reaction'],
))

# Lignes de seuil
fig5.add_hline(y=2, line=dict(color=COLORS['accent'], dash='dash', width=1.5),
               annotation_text='Seuil PRR=2', annotation_font_color=COLORS['accent'])
fig5.add_vline(x=3, line=dict(color='#8B949E', dash='dot', width=1),
               annotation_text='n min=3', annotation_font_color='#8B949E')

fig5.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title='🔍 Détection de Signaux Pharmacovigilance — VEDOLIZUMAB (FAERS 2025)<br>'
          '<sup>716 signaux confirmés sur 2524 réactions analysées (PRR≥2, Chi²≥4, ROR IC95inf>1)</sup>',
    xaxis=dict(title='Nombre de rapports (n)', type='log',
               gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
    yaxis=dict(title='PRR (Proportional Reporting Ratio)', type='log',
               gridcolor=COLORS['grid'], zerolinecolor=COLORS['grid']),
    legend=dict(x=0.7, y=0.95),
    height=680,
    margin=dict(t=90, l=80, r=40, b=60),
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=1, y=-0.1, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='right'
    )]
)
fig5.show()
save_and_download(fig5, 'linkedin_05_signals_vedolizumab.png', width=1300, height=720)


# ============================================================
# 📊 VIZ 6 — Dashboard résumé (4 panels en 1 image)
# ============================================================
print('\nCréation Viz 6 — Dashboard résumé...')

df_sex     = pd.read_csv('viz_sex.csv')
df_age     = pd.read_csv('viz_age_groups.csv').sort_values('count', ascending=True)
df_quarter = pd.read_csv('viz_quarters.csv')
df_top5    = pd.read_csv('viz_top_drugs.csv').head(10).sort_values('count', ascending=True)

fig6 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '👤 Répartition par Sexe',
        '📅 Rapports par Quarter',
        '🎂 Groupe d\'âge',
        '💊 Top 10 Médicaments Suspects',
    ),
    specs=[
        [{'type': 'pie'},    {'type': 'xy'}],
        [{'type': 'xy'},     {'type': 'xy'}]
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.10,
)

# Panel 1 : Sexe (donut)
sex_colors = [COLORS['secondary'], COLORS['primary'], '#8B949E']
fig6.add_trace(go.Pie(
    labels=df_sex['sex_label'], values=df_sex['count'],
    hole=0.5, marker_colors=sex_colors,
    textinfo='label+percent', textfont_size=11,
    showlegend=False,
    hovertemplate='<b>%{label}</b>: %{value:,}<extra></extra>',
), row=1, col=1)

# Panel 2 : Quarters (bar)
fig6.add_trace(go.Bar(
    x=df_quarter['quarter'], y=df_quarter['count'],
    marker_color=COLORS['primary'],
    hovertemplate='%{x}: %{y:,} rapports<extra></extra>',
    showlegend=False,
), row=1, col=2)

# Panel 3 : Âge (bar horizontal)
fig6.add_trace(go.Bar(
    x=df_age['count'], y=df_age['age_group'],
    orientation='h',
    marker_color=COLORS['purple'],
    hovertemplate='<b>%{y}</b>: %{x:,}<extra></extra>',
    showlegend=False,
), row=2, col=1)

# Panel 4 : Top 10 médicaments (bar horizontal)
fig6.add_trace(go.Bar(
    x=df_top5['count'], y=df_top5['drugname'],
    orientation='h',
    marker=dict(
        color=df_top5['count'],
        colorscale=[[0, '#0f3460'], [1, '#FFD93D']],
    ),
    hovertemplate='<b>%{y}</b>: %{x:,}<extra></extra>',
    showlegend=False,
), row=2, col=2)

fig6.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title=dict(
        text='📊 FAERS 2025 — Tableau de Bord Pharmacovigilance<br>'
             '<sup>1.6M rapports · Q1-Q4 2025 · FDA Adverse Event Reporting System</sup>',
        font=dict(size=18, color=COLORS['primary']),
    ),
    height=800,
    margin=dict(t=100, l=60, r=40, b=60),
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | Analyse : Python · Pandas · Plotly | @VotreNom',
        x=0.5, y=-0.06, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='center'
    )]
)

# Style axes
for row, col in [(1,2),(2,1),(2,2)]:
    fig6.update_xaxes(gridcolor=COLORS['grid'], row=row, col=col)
    fig6.update_yaxes(gridcolor=COLORS['grid'], row=row, col=col)

fig6.show()
save_and_download(fig6, 'linkedin_06_dashboard.png', width=1400, height=850)

print('\n🎉 Toutes les visualisations sont téléchargées !')
print('\n📋 Récapitulatif des fichiers LinkedIn :')
print('  linkedin_01_treemap_drugs.png      → Slide 1 : Médicaments suspects')
print('  linkedin_02_top_reactions.png      → Slide 2 : Réactions adverses')
print('  linkedin_03_outcomes.png           → Slide 3 : Gravité')
print('  linkedin_04_world_map.png          → Slide 4 : Carte mondiale')
print('  linkedin_05_signals_vedolizumab.png → Slide 5 : Signal detection')
print('  linkedin_06_dashboard.png          → Slide 6 : Dashboard résumé')

# ============================================================
# 📊 VIZ 7 — Bar chart : Top 20 Indications
# ============================================================

# Installer kaleido pour l'exportation d'images si ce n'est pas déjà fait
!pip install kaleido -q

# Force reload plotly.io to ensure it detects newly installed kaleido
import importlib
import plotly.io as pio
importlib.reload(pio)

print('\nCréation Viz 7 — Top Indications...')

# Prepare data for top indications
top_indications = df_indi['indi_pt'].value_counts().head(20).sort_values(ascending=True)

fig7 = go.Figure()
fig7.add_trace(go.Bar(
    x=top_indications.values,
    y=top_indications.index,
    orientation='h',
    marker=dict(
        color=top_indications.values,
        colorscale=[[0, '#0f3460'], [1, '#FFD93D']],
        line=dict(color=COLORS['bg'], width=0.5),
    ),
    hovertemplate='<b>%{y}</b><br>Rapports: %{x:,}<extra></extra>',
))
fig7.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title_text='📈 Top 20 Indications Signalées — FAERS 2025',
    xaxis_title='Nombre de rapports',
    yaxis_title='',
    margin=dict(t=70, l=280, r=40, b=50),
    height=650,
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=1, y=-0.08, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='right'
    )]
)
fig7.show()
save_and_download(fig7, 'linkedin_07_top_indications.png', width=1200, height=680)

# ============================================================
# 📊 VIZ 8 — Pie chart : Routes d'administration des médicaments
# ============================================================

# Installer kaleido pour l'exportation d'images si ce n'est pas déjà fait
!pip install kaleido -q

# Force reload plotly.io to ensure it detects newly installed kaleido
import importlib
import plotly.io as pio
importlib.reload(pio)

print('\nCréation Viz 8 — Routes d\'administration...')

# Prepare data for drug routes, filter out 'Unknown' and 'nan'
route_counts = data_demo_drug['route'].value_counts().head(10)
route_counts = route_counts[route_counts.index != 'Unknown']
route_counts = route_counts[route_counts.index != 'nan']

fig8 = go.Figure(go.Pie(
    labels=route_counts.index,
    values=route_counts.values,
    hole=0.4,
    marker=dict(
        colors=px.colors.sequential.Plotly3,
        line_color=COLORS['bg'],
        line_width=3
    ),
    hovertemplate='<b>%{label}</b><br>%{value:,} rapports<br>%{percent}<extra></extra>',
    textinfo='percent',
    textfont=dict(size=12),
))
fig8.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    title='💉 Top 10 Routes d\'Administration des Médicaments — FAERS 2025',
    annotations=[
        dict(text='Routes<br>Total', x=0.5, y=0.5, font_size=18,
             showarrow=False, font=dict(color=COLORS['text']))
    ],
    height=600,
    showlegend=True,
    legend=dict(x=0.9, y=0.5),
)
fig8.show()
save_and_download(fig8, 'linkedin_08_drug_routes.png', width=1100, height=650)

# ============================================================
# 📊 VIZ 9 — Histogramme : Distribution par âge des patients
# ============================================================

# Installer kaleido pour l'exportation d'images si ce n'est pas déjà fait
!pip install kaleido -q

# Force reload plotly.io to ensure it detects newly installed kaleido
import importlib
import plotly.io as pio
importlib.reload(pio)

print('\nCréation Viz 9 — Distribution par âge...')

# Filter out NaN and create age groups if necessary, or use directly
age_data = data_demo_drug['patientage'].dropna()

fig9 = px.histogram(
    age_data,
    nbins=30, # Adjust number of bins for desired granularity
    title='🎂 Distribution par Âge des Patients — FAERS 2025',
    color_discrete_sequence=[COLORS['purple']],
)
fig9.update_layout(
    **TEMPLATE['layout'].to_plotly_json(),
    xaxis_title='Âge du Patient',
    yaxis_title='Nombre de Rapports',
    bargap=0.1,
    annotations=[dict(
        text='Source: FDA FAERS Q1-Q4 2025 | @VotreNom',
        x=0.5, y=-0.15, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='#8B949E'), xanchor='center'
    )]
)
fig9.show()
save_and_download(fig9, 'linkedin_09_patient_age_distribution.png', width=1200, height=700)

pip install --upgrade plotly==6.1.1 -q

"""Now that Plotly has been upgraded, I will re-run the cell containing the visualizations to generate and download the images."""